# Lekcija 18: Osiguravanje AI agenata pomoću kriptografskih potvrda

## Praktična bilježnica

Ova bilježnica vodi kroz četiri vježbe:

1. **Potpiši svoju prvu potvrdu** za poziv alata agenta i verificiraj je.
2. **Izmijeni potvrdu** i promatraj kako provjera ne uspije.
3. **Izgradi lanac od tri potvrde** i potvrdi integritet lanca.
4. **Omotaj poziv alata iz Microsoft Agent Frameworka** tako da svaka radnja izda potvrdu.

Sve kriptografske primitivne funkcije su uvezene iz dobro održavanih biblioteka (`pynacl` za Ed25519, `jcs` za RFC 8785 kanonski JSON, `hashlib` iz standardne Python biblioteke za SHA-256). Sama logika potvrde je običan Python koji možete čitati i mijenjati.

Pokrenite ćelije redom. Svaki odjeljak je kratak i samostalan.


## Postavljanje

Instalirajte dvije ovisnosti. Obje imaju permisivne licence (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Pomoćni alati

Ova dva pomoćna alata obrađuju base64url kodiranje (bez paddinga) i SHA-256 hashiranje proizvoljnih objekata. Ostatak bilježnice fokusira se na samu logiku primitka.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Poglavlje 1: Potpišite svoj prvi račun

Zamislite da je naš agent za **Contoso Travel** upravo potražio letove iz Sydneya za Los Angeles za korisnika. Želimo zabilježiti izjavu gatewaya o ovom pozivu alata kako bi budući revizor mogao potvrditi istu pod zasebno postavljenim ključem izdavatelja.

### Korak 1.1: Generirajte ključ za potpisivanje

U produkciji bi potpisni ključ agenta bio pohranjen u hardverskom sigurnosnom modulu (HSM), Azure Key Vault-u ili sličnoj zaštićenoj pohrani. Za ovu lekciju generiramo novi ključ u memoriji.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Korak 1.2: Izradi sadržaj potvrde

Sadržaj sadrži sve što želimo da potvrda potvrdi: tko je djelovao, koji je alat korišten, s kojim argumentima, što se vratilo, prema kojoj politici i kada. Umjesto da ih uključimo izravno, heširamo argumente i rezultat kako potvrda ne bi otkrivala osjetljive podatke.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Korak 1.3: Potpišite i sastavite potvrdu

Tri koraka:

1. Kanonizirajte korisničke podatke koristeći JCS kako bi dvije implementacije koje proizvode istu logičku potvrdu proizvele identične bajtove.
2. Direktno potpišite kanonizirane bajtove Ed25519 privatnim ključem. PureEdDSA interno hešira poruku, pa bi dodatno prethodno heširanje promijenilo protokol.

Potpis se zatim prilaže izvornoj korisničkoj poruci radi stvaranja konačne potvrde.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Korak 1.4: Potvrdite primitak

Potvrda je obrnuti proces. Uklanjamo potpis, ponovo izračunavamo kanonske bajtove i provjeravamo potpis u odnosu na javni ključ izdavatelja.

Potvrda je offline: nema poziva servisa niti pristupa mreži. Ipak treba jednu stvar osim primitka: javni ključ izdavatelja, dobiven putem kanala kojem revizor već vjeruje, kao što je JWK Set koji izdavatelj objavljuje ili unaprijed ušteđenu vrijednost. Polje `public_key` unutar primitka ne može igrati tu ulogu, jer tko god piše primitak, također piše i to polje. Odjeljak 2 prikazuje krivotvorenje koje se time sprječava.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Trebali biste vidjeti `Receipt is valid: True`, `Pre-hashed receipt valid: False` i `Malformed receipts rejected: True`. Pozitivan slučaj dokazuje da izravni-JCS put potpisivanja radi; negativna kontrola čini pravilo opsega potpisa izvršnim umjesto da ostane kao proza; zadnji redak prikazuje krivo oblikovani unos koji vraća `False` umjesto da izbaci iznimku.


## Odjeljak 2: Manipuliranje računom

Cijela poanta računa je da budu otporni na manipulacije. Dokazat ćemo to.

Izmijenit ćemo točno jedan znak na računu i pratiti neuspjeh provjere.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Što se upravo dogodilo?

Kada smo promijenili `policy_id`, kanonski bajtovi su se promijenili. Potpis (koji je bio nad izvornim kanonskim bajtovima) više se ne podudara. Verifikacija ispravno vraća `False`.

Nema načina da se promijeni potpisano polje, tj. bilo što izvan `signature` objekta, a da se i dalje potvrdi primitak protiv javnog ključa izdavatelja, osim ako napadač nema privatni ključ izdavatelja. Sam `signature` objekt nije potpisan: `sig` je potpis, a `verify_receipt` zanemaruje `public_key`, tako da uređivanje tog polja samo po sebi ne mijenja ništa. Sve dok je privatni ključ u skladištu ključeva, a verificeri koriste izdavateljev neovisno objavljeni ključ, promjena potpisanog polja je nemoguća za sakriti. Taj je uvjet bitan: sljedeća ćelija pokazuje modificirani primitak koji se savršeno potvrđuje protiv drugačijeg ključa.

Isprobajte sami: izmijenite `tool_name` ili `agent_id` ili `timestamp` u gornjoj ćeliji i pokrenite ponovno. Svaka promjena potpisanog polja rezultira nevaljanim primitkom.


### Neovlaštena izmjena koju potpis ne može otkriti

Napadač iznad nije ponovno potpisao. Pažljiviji bi to učinio: izmijenio bi potvrdu,
potpisao je ključem koji je sam generirao i stavio taj ključ u
`public_key` polje. Rezultat je potvrda čiji je potpis u potpunosti autentičan
za njen sadržaj.

Pokrenite ovo i pogledajte koja od dvije provjere to otkriva.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Trebali biste vidjeti `True` zatim `False`. Ta tri reda `assert` čine to
provjerom: ćelija baca pogrešku ako krivotvorenje ikada potvrdi valjanost u odnosu na
izdavateljev ključ.

Krivotvorenje je unutarnje konzistentno. Njegov potpis stvarno odgovara njegovom
sadržaju i ključu koji nosi, pa provjeritelj koji pročita ključ iz
potvrde prijavljuje ga kao valjanu. Ništa na potvrdi ne izgleda pogrešno, jer
ništa na njoj nije nekonzistentno.

Jedino što ga otkriva je provjera u odnosu na ključ na koji napadač nije imao
utjecaj. Ovo je razlog zašto `verify_receipt` prima ključ kao argument: funkcija
sa potpisom `verify_receipt(receipt)` ne može biti sigurno napisana,
jer je jedini ključ do kojeg može doći onaj koji je napadač dostavio.

Ista potvrda je spremljena kao `sample_receipts/04_resigned_receipt.json` ako želite
pregledati izvan bilježnice.


## Odjeljak 3: Povezivanje potvrda u lanac

Jedna potvrda štiti jednu radnju. Većina agenata obavlja mnogo radnji. Kako bismo cijeli niz učinili otpornim na neovlaštene izmjene, povezujemo svaku potvrdu s prethodnom uključivanjem hash vrijednosti prethodne potvrde u sadržaj nove potvrde.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Ako netko ukloni ili promijeni redoslijed potvrda u danom nizu, sljedeća veza će biti prekinuta. Za otkrivanje skraćivanja ili uređivanja sufiksa kojeg kontrolira potpisnik također je potrebna fiksirana trenutačna glava, vanjska kontrolna točka ili autentificirana predaja populacije.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Sada prekini lanac mijenjajući srednji račun i ponovno provjeri. Promijenjeni račun pada provjeru potpisa, I sljedeći račun pada provjeru veze lanca (jer njegov `previous_receipt_hash` više ne odgovara hash vrijednosti promijenjenog srednjeg računa).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Potvrda 0 i dalje se provjerava (nije izmijenjena i nema prethodnika na kojem ovisi). Potvrda 1 ne uspijeva provjeru potpisa jer smo promijenili `tool_args_hash`. Potvrda 2 ne uspijeva provjeru lančane veze jer je njezin `previous_receipt_hash` izračunat na temelju originalne (sada izmijenjene) potvrde 1.

Čak i ako napadač ponovno potpiše izmijenjenu potvrdu 1, novi potpis ne uspijeva provjeru s ključem izdavatelja osim ako ne posjeduje privatni ključ izdavatelja (ponovni potpis njihovim vlastitim ključem ne uspijeva, kao što je pokazao Odjeljak 2), a nesklad lančane veze u potvrdi 2 i dalje bi otkrio manipulaciju. Da bi prikrio promjenu, napadač bi morao ponovno potpisati svaku potvrdu od točke izmjene nadalje, što zahtijeva posjedovanje privatnog ključa izdavatelja.


## Poglavlje 4: Omotajte poziv alata agenta s potpisivanjem primitka

U stvarnoj implementaciji, ne želite da svaki autor agenta pamti pozvati `make_receipt`. Želite da potpisivanje primitka bude automatsko za svaki poziv alata.

Ovo je najjednostavniji obrazac: omotna klasa koja prima bilo koju pozivu funkciju alata i vraća verziju koja emitira primitak. Ovo se prilagođava bilo kojem okviru agenta, uključujući Microsoft Agent Framework (`agent_framework.foundry`).

Ako nemate postavljen Microsoft Foundry projekt, lokalni mock ispod i dalje prikazuje obrazac.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integracija s Microsoft Agent Frameworkom

Gornji `ReceiptedTool` omotač ne ovisi o frameworku. Da biste ga koristili unutar agenta izgrađenog pomoću Microsoft Agent Frameworka, registrirajte omotanu funkciju kao alat. Skica (zamijenili biste lažni alat stvarnom registracijom Microsoft Foundry alata):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Agent framework ne treba ništa znati o potvrđivanju. Potpisivanje potvrda je omotano oko alata, nije ugrađeno u framework. Ovo je način na koji dodajete podrijetlo postojećem kodu agenta bez prepisivanja agenta.


## Sažetak i izazov za proširenje

Imate:

- Generirani Ed25519 par ključeva.
- Izgrađen i potpisan račun za poziv alata agenta.
- Potvrđen račun izvan mreže koristeći samo javni ključ izdavača, dobiven neovisno o računu.
- Manipulirano s računom i uočen neuspjeh provjere.
- Izgrađena sekvenca od tri računa povezane hash lancem.
- Manipulirano s sredinom lanca i uočen neuspjeh potpisa i neuspjeh veze u lancu.
- Funkcija alata omotana automatskim potpisivanjem računa.

**Izazov za proširenje.** Proširite shemu računa s `request_id` poljem (UUID za distribuirano trasiranje). Ažurirajte `make_receipt` da ga uključi i potvrdite da se računi i dalje uspješno potvrđuju od početka do kraja. Zatim modificirajte polje nakon potpisivanja i potvrdite da provjera ne uspijeva. Ovo vas prisiljava da interno shvatite kako svaki bajt kanonskog kodiranja doprinosi potpisu.

**Važna granica.** Pod kontekstom zalijepljenog ključa i lanca, računi dokazuju atribuciju potpisa, integritet sadržaja i vezu s predstavljenim prethodnikom. Oni NE dokazuju izvršenje, istinu izvora, pouzdano vrijeme, potpunost, procjenu politike ili ispravnost. Računi su temelj. Upravljanje je sustav koji gradite iznad toga.

Ponovno pročitajte README lekcije imajući tu granicu na umu. Najčešća pogreška koju timovi rade s računima je pretpostavka da "imamo račune" znači "upravljaju nama". Ne znači. Računi čine ponašanje agenta audibilnim. Ne čine ga ispravnim.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
